# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [ ]:
# Load environment
%load_ext dotenv
%dotenv


In [42]:
# Import Dask
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.
(1pt)

In [43]:
# Import glob
import os
from glob import glob

# Set directory
price_data_dir = os.getenv("PRICE_DATA")
# Look through sub-directories for parquet files
parquet_files = glob(os.path.join(price_data_dir, "**", "*.parquet"), recursive=True)

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [44]:
# create Dask data frame from parquet files
ddf = dd.read_parquet(parquet_files).set_index("Ticker")

# # add lags for Close and Adj_Close, grouped by Ticker
def add_lags(df):
    df['Close_lag_1'] = df.groupby('Ticker')['Close'].shift(1)
    df['Adj_Close_lag_1'] = df.groupby('Ticker')['Adj Close'].shift(1)
    return df
ddf = ddf.map_partitions(add_lags)

# Calculate returns
ddf["returns"] = (ddf["Close"] / ddf["Close_lag_1"]) - 1

# Calculate range
ddf["hi_lo_range"] = ddf["High"] - ddf["Low"]

# Store as dd_feat
dd_feat = ddf

print(dd_feat.head())

Price        Date  Adj Close      Close       High        Low       Open  \
Ticker                                                                     
A      2000-01-03        NaN  43.382828  47.562942  40.596082  47.449965   
A      2000-01-04        NaN  40.068878  41.499910  39.014433  41.048003   
A      2000-01-05        NaN  37.583420  40.068898  36.340684  39.918261   
A      2000-01-06        NaN  36.152363  37.357444  35.022601  37.131491   
A      2000-01-07        NaN  39.165062  39.729943  35.549827  35.587484   

Price      Volume  Year  Close_lag_1  Adj_Close_lag_1   returns  hi_lo_range  
Ticker                                                                        
A       4674353.0  2000          NaN              NaN       NaN     6.966860  
A       4765083.0  2000    43.382828              NaN -0.076389     2.485477  
A       5758642.0  2000    40.068878              NaN -0.062030     3.728214  
A       2534434.0  2000    37.583420              NaN -0.038077     2.33

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [45]:
# Convert Dask data frame to pandas data frame
df_pandas = dd_feat.compute()

# Add 10 day rolling average for returns, grouped by Ticker
df_pandas['returns_moving_avg'] = df_pandas.groupby('Ticker')['returns'].rolling(window=10).mean().reset_index(level=0, drop=True)
print(df_pandas.iloc[10:20])

Price        Date  Adj Close      Close       High        Low       Open  \
Ticker                                                                     
A      2000-01-18        NaN  43.081573  43.910065  41.048001  41.349271   
A      2000-01-19        NaN  42.177765  43.382847  42.064791  43.382847   
A      2000-01-20        NaN  41.047997  42.704982  40.671409  42.629662   
A      2000-01-21        NaN  41.424599  41.801186  40.031226  41.650552   
A      2000-01-24        NaN  41.273956  43.269869  40.972686  41.273956   
A      2000-01-25        NaN  40.784382  41.951803  40.031208  41.424579   
A      2000-01-26        NaN  41.160980  41.725861  40.897370  40.935030   
A      2000-01-27        NaN  41.236294  42.479032  40.520779  42.102445   
A      2000-01-28        NaN  41.010349  41.876501  40.633762  41.876501   
A      2000-01-31        NaN  39.880589  40.746740  39.014437  40.709080   

Price      Volume  Year  Close_lag_1  Adj_Close_lag_1   returns  hi_lo_range  \
Ticker 

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

No it was not necessary to convert to Pandas, because Dask supports rolling operations.

It may have been better to calculate moving average in Dask if the dataset was very large and we wanted to verify that the moving average calculation was working without computing the entire dataset.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.